In [6]:
import pandas as pd
import plotly.express as px
import streamlit_authenticator as stauth
import urllib
import mysql.connector
import csv
from sqlalchemy import create_engine, text

In [4]:
player_df = pd.read_csv('data files/player_with_credentials.csv')
names = player_df['Player'].to_list()
usernames = player_df['app_username'].tolist()
passwords = player_df['app_password'].tolist()
playerids = player_df['playerID'].tolist()
positionids = player_df['positionID'].tolist()

In [7]:
credentials = {"usernames": {}}
for name, playerid, positionid, username, password in zip(names, playerids, positionids, usernames, passwords):
    credentials["usernames"][username] = {
        "name": name,
        "playerID" : playerid,
        "positionID" : positionid,
        "password": password
    }

In [8]:
username = 'CJDaniels'

In [9]:
playerid = player_df.loc[player_df['app_username'] == username]['playerID'].iloc[0]
positionid = player_df.loc[player_df['app_username'] == username]['positionID'].iloc[0]
print(playerid, positionid)

287 10.0


In [12]:
hashed_passwords = stauth.Hasher.hash_passwords(credentials=credentials)

In [13]:
with open("hashed_credentials osu.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    # Header
    writer.writerow(["username", "name", "playerID", "password"])
    
    # Rows
    for username, details in credentials["usernames"].items():
        writer.writerow([username, details["name"], details["playerID"], details["password"]])

In [14]:
cmj_df = pd.read_csv('data files/cmj.csv')
cmj_df = cmj_df[['date', 'playerID', 'Jump Height']]
cmj_df = cmj_df.loc[cmj_df['playerID'] == playerid]

cmj_df = cmj_df.sort_values(['date'])
player_peak_cmj = cmj_df.sort_values(['Jump Height']).tail(1)
player_peak_cmj_jump = player_peak_cmj['Jump Height'].iloc[0]
player_peak_cmj_date = player_peak_cmj['date'].iloc[0]
cmj_df = cmj_df.groupby(['date']).mean().reset_index()
cmj_df

,date,playerID,Jump Height
0,2025-04-04,287.0,0.4750
1,2025-04-07,287.0,0.4850
2,2025-04-21,287.0,0.5000
3,2025-05-19,287.0,0.5000
4,2025-05-23,287.0,0.5350
5,2025-05-26,287.0,0.4850
6,2025-06-02,287.0,0.5000
7,2025-06-06,287.0,0.5100
8,2025-06-09,287.0,0.5400
9,2025-06-13,287.0,0.5200


In [ ]:
## Position Force Plate Data (2025)
position_cmj_df = pd.read_csv('data files/cmj.csv')
merge_player = pd.read_csv('data files/player_with_credentials.csv')
merge_position = pd.read_csv('data files/positions.csv')
position_cmj_df = position_cmj_df.merge(merge_player, on='playerID')
position_cmj_df = position_cmj_df.merge(merge_position, on='positionID')
position_cmj_df = position_cmj_df.loc[position_cmj_df['positionID'] == positionid]
position_cmj_df = position_cmj_df.loc[position_cmj_df['date'] > '2025-01-01']
position_cmj_df = position_cmj_df[['playerID', 'Position', 'date', 'Jump Height']]

raw_position_cmj_df = position_cmj_df.sort_values(['date'])
## Storing position in variable to remove column
position_name = raw_position_cmj_df.tail(1)['Position'].iloc[0]
position_cmj_df = raw_position_cmj_df.groupby(['date', 'playerID']).max(numeric_only=True).reset_index()
position_cmj_df

,date,playerID,Jump Height
0,2025-01-11,288,0.47
1,2025-01-11,289,0.40
2,2025-01-13,78,0.48
3,2025-01-13,130,0.56
4,2025-01-13,147,0.54
...,...,...,...
628,2026-02-02,327,0.55
629,2026-02-02,338,0.41
630,2026-02-02,349,0.54
631,2026-02-02,355,0.61


In [18]:
raw_position_cmj_df.groupby('date').mean(numeric_only=True).reset_index()[['date','Jump Height']]

,date,Jump Height
0,2025-01-11,0.412500
1,2025-01-13,0.474783
2,2025-01-20,0.462727
3,2025-01-27,0.470000
4,2025-02-03,0.486000
...,...,...
60,2026-01-14,0.445000
61,2026-01-22,0.475833
62,2026-01-26,0.470000
63,2026-01-30,0.494500


In [20]:
position_cmj_df = position_cmj_df.sort_values(['Jump Height']).tail(1)
highest_position_jump = position_cmj_df['Jump Height'].iloc[0]
highest_position_date = position_cmj_df['date'].iloc[0]
highest_position_player = position_cmj_df['playerID'].iloc[0]
highest_position_player = player_df.loc[player_df['playerID'] == highest_position_player]['Player'].iloc[0]
split_name = highest_position_player.split()
highest_position_player = " ".join([split_name[-1]] + split_name[:-1])
print(highest_position_player, highest_position_date, highest_position_jump)

split_name = highest_position_player.split()
highest_position_player = " ".join([split_name[-1]] + split_name[:-1])

Joshisa Trader 2025-06-09 0.68


In [31]:
catapult_df = pd.read_csv('data files/catapult.csv')
catapult_df = catapult_df.loc[catapult_df['playerID'] == playerid]
catapult_df = catapult_df[['playerID', 'Date', 'Total Player Load', 'Maximum Velocity']]

catapult_df.groupby(['Date']).agg({'Total Player Load' : 'sum', 'Maximum Velocity' : 'max'}).reset_index().sort_values(['Date'])

,Date,Total Player Load,Maximum Velocity
0,2025-03-11,246.27,10.90
1,2025-03-13,362.00,13.46
2,2025-03-17,63.87,13.11
3,2025-03-18,198.40,5.86
4,2025-03-20,323.14,16.79
...,...,...,...
172,2026-01-15,370.85,17.77
173,2026-01-16,347.99,18.41
174,2026-01-17,317.39,17.92
175,2026-01-18,187.82,11.20


In [28]:
body_weight_df = pd.read_csv('data files/body_weight.csv')
## This dropna is new by the way
body_weight_df = body_weight_df.dropna(subset='Weight')
body_weight_df = body_weight_df.loc[body_weight_df['playerID'] == playerid]
body_weight_df = body_weight_df[['playerID', 'date', 'Weight', 'Ideal Weight', 'BF%']]

body_weight_df = body_weight_df.sort_values(['date'])
body_weight_df

,playerID,date,Weight,Ideal Weight,BF%
16254,287,2025-01-12,203.0,202.5,NaN
19740,287,2025-02-24,204.0,202.5,NaN
19739,287,2025-02-25,205.0,202.5,NaN
20028,287,2025-02-26,204.0,202.5,NaN
20029,287,2025-02-27,207.0,202.5,NaN
...,...,...,...,...,...
39700,287,2026-01-12,203.0,202.5,10.0
39863,287,2026-01-14,201.0,202.5,10.0
39925,287,2026-01-15,203.0,202.5,10.0
40165,287,2026-01-16,202.0,202.5,10.0


In [29]:
## Baseline Data
latest_weight = body_weight_df.tail(1)['Weight'].iloc[0]
latest_bf = body_weight_df.tail(1)['BF%'].iloc[0]

data = {'Metric' : ['Body Weight (lbs)', 'Body Fat (%)'], 'Value' : [latest_weight, latest_bf]}
baseline_df = pd.DataFrame(data)
baseline_df

,Metric,Value
0,Body Weight (lbs),203.0
1,Body Fat (%),NaN


In [32]:
internal_df = pd.read_csv('data files/internalload2.csv')
internal_df = internal_df.loc[internal_df['playerID'] == playerid]
internal_df = internal_df[['playerID', 'Practice Date', 'SDNN', 'Calories Burned']]
internal_df

,playerID,Practice Date,SDNN,Calories Burned
8,287,2025-06-10,126.05,2131.22
60,287,2025-06-12,143.67,1996.44
110,287,2025-06-13,122.91,1084.43
281,287,2025-06-23,130.43,1316.01
282,287,2025-06-24,144.28,2175.81
...,...,...,...,...
7784,287,2026-01-12,123.73,955.61
7842,287,2026-01-15,126.02,1761.94
7900,287,2026-01-16,118.02,1374.15
7955,287,2026-01-17,139.73,958.50


In [33]:
internal_df.mean(numeric_only=True)['Calories Burned']

np.float64(1221.2062585034014)

In [34]:
perch_df = pd.read_csv('data files/perch.csv')
perch_df = perch_df.loc[perch_df['playerID'] == playerid]
perch_df = perch_df[['playerID', 'date', 'exerciseID', 'Weight (lbs)', 'Set Avg Peak Velocity (m/s)']]

bench_data = perch_df.loc[perch_df['exerciseID'] == 1]

player_peak_bench = bench_data.sort_values(['Weight (lbs)']).tail(1)
player_peak_bench_rep = player_peak_bench['Weight (lbs)'].iloc[0]
player_peak_bench_date = player_peak_bench['date'].iloc[0]

bench_data = bench_data.groupby('date').mean(numeric_only=True).reset_index()

In [35]:
player_bench_max = perch_df.loc[perch_df['exerciseID'] == 1].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]
player_bench_max_date = perch_df.loc[perch_df['exerciseID'] == 1].sort_values(['Weight (lbs)']).tail(1)['date'].iloc[0]
player_squat_max = perch_df.loc[perch_df['exerciseID'] == 4].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]
player_squat_max_date = perch_df.loc[perch_df['exerciseID'] == 4].sort_values(['Weight (lbs)']).tail(1)['date'].iloc[0]
player_power_clean_max = perch_df.loc[perch_df['exerciseID'] == 9].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]
player_power_clean_max_date = perch_df.loc[perch_df['exerciseID'] == 9].sort_values(['Weight (lbs)']).tail(1)['date'].iloc[0]
print(player_power_clean_max, player_power_clean_max_date)

275 2025-07-14


In [36]:
perch_df.loc[perch_df['exerciseID'] == 9].sort_values(['Weight (lbs)']).tail(1)['Weight (lbs)'].iloc[0]

np.int64(275)

In [12]:
hydration_df = pd.read_csv('data files/hydration.csv')
merge_status = pd.read_csv('data files/hydration_status.csv')
hydration_df = hydration_df.merge(merge_status, on='hydration_statusID')
hydration_df = hydration_df.loc[hydration_df['playerID'] == playerid]
df = hydration_df[['playerID', 'Test Date', 'mOsm', 'Hydration Status']]